### [Usual ML Metrics Don’t Work for Decision Models](https://medium.com/data-science-collective/why-your-usual-ml-metrics-dont-work-for-decision-models-f2fd9d590f34)

> A practical guide to evaluating counterfactual predictions with matching techniques

In [1]:
%pip install -q datasets catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.0 MB/s eta 0:00:00


In [2]:
from catboost import CatBoostRegressor
from datasets import load_dataset
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [3]:
def is_numeric(x):
    try:
        x + 1
        return True
    except:
        return False


def beautify_int(x):
    if type(x) in [bool, np.bool]:
        return str(x)
    if x is None or np.isnan(x):
        return ""
    try:
        return f"{int(x):,.0f}"
    except:
        return str(x)

In [4]:
df_example = pd.DataFrame({
    "SquareFeet (Covariate)": [600, 600, 1_700, 1_700],
    "OverallCondition (Decision)": [4, 7, 5, 8],
    "SalePrice (Target)": [88_000, 95_000, 247_000, 271_000],
    "ModelPrediction": [90_000, 90_000, 260_000, 260_000],
})

df_example.map(beautify_int)

,SquareFeet (Covariate),OverallCondition (Decision),SalePrice (Target),ModelPrediction
0,600,4,"88,000","90,000"
1,600,7,"95,000","90,000"
2,"1,700",5,"247,000","260,000"
3,"1,700",8,"271,000","260,000"


In [5]:
dataset = load_dataset("ttd22/house-price")

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [6]:
df = dataset["train"].to_pandas().sample(frac=1, replace=False)
df.columns = [("HouseId" if c == "Id" else c) for c in df.columns]
cat_columns = list(df.columns[~df.apply(is_numeric)])
df[cat_columns] = df[cat_columns].fillna("void")

In [7]:
df.sample(15)

,HouseId,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
961,962,60,RL,NaN,12227,Pave,void,IR1,Lvl,AllPub,...,0,void,void,void,0,7,2008,WD,Normal,272000
524,525,60,RL,95.0,11787,Pave,void,IR1,Lvl,AllPub,...,0,void,void,void,0,8,2007,WD,Normal,315750
519,520,70,RL,53.0,10918,Pave,void,Reg,Lvl,AllPub,...,0,void,MnPrv,void,0,6,2009,WD,Normal,234000
631,632,120,RL,34.0,4590,Pave,void,Reg,Lvl,AllPub,...,0,void,void,void,0,8,2007,WD,Normal,209500
981,982,60,RL,98.0,12203,Pave,void,IR1,Lvl,AllPub,...,0,void,void,void,0,7,2009,WD,Normal,336000
1171,1172,20,RL,76.0,9120,Pave,void,Reg,Lvl,AllPub,...,0,void,void,Shed,1400,11,2008,WD,Normal,163000
521,522,20,RL,90.0,11988,Pave,void,IR1,Lvl,AllPub,...,0,void,void,void,0,5,2007,WD,Normal,150000
1416,1417,190,RM,60.0,11340,Pave,void,Reg,Lvl,AllPub,...,0,void,void,void,0,4,2010,WD,Normal,122500
321,322,60,RL,99.0,12099,Pave,void,IR1,Lvl,AllPub,...,0,void,void,void,0,6,2007,WD,Normal,354000
1393,1394,190,RM,60.0,10800,Pave,Pave,Reg,Lvl,AllPub,...,0,void,void,void,0,4,2008,WD,Normal,163000


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1460 entries, 892 to 1126
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   HouseId        1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          1460 non-null   object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallCond

In [9]:
target = "SalePrice"

decision_variables = ["OverallCond"]

monotone_constraints = {
    "OverallCond": 1,
}

covariates = [
    "LotArea", "MasVnrArea", "GrLivArea", "GarageArea",
    "PoolArea", "1stFlrSF", "2ndFlrSF", "TotalBsmtSF",
    "Foundation", "YearBuilt", "Neighborhood",
]

full_model_prediction = "FullModel"

covariate_model_prediction = "CovariateModel"

full_model_cat_features = [c for c in cat_columns if c in covariates + decision_variables]

covariate_model_cat_features = [c for c in cat_columns if c in covariates]

In [10]:
%%time

full_models = []
covariate_models = []

# Because we don't have many data points, we train 10 models with a 80/20 split
for ix_model in tqdm(range(10)):

    ix_trn, ix_tst = train_test_split(df.index, test_size=.20)

    full_model = CatBoostRegressor(
        monotone_constraints=monotone_constraints,
        silent=True
    ).fit(
        X=df.loc[ix_trn, covariates + decision_variables],
        y=df.loc[ix_trn, target],
        cat_features=full_model_cat_features,
    )

    covariate_model = CatBoostRegressor(
        silent=True,
    ).fit(
        X=df.loc[ix_trn, covariates],
        y=df.loc[ix_trn, target],
        cat_features=covariate_model_cat_features,
    )

    full_models.append(full_model)
    covariate_models.append(covariate_model)

    df.loc[ix_tst, f"IsTest_{ix_model}"] = True
    df.loc[ix_tst, f"{full_model_prediction}_{ix_model}"] = full_model.predict(df.loc[ix_tst, full_model.feature_names_])
    df.loc[ix_tst, f"{covariate_model_prediction}_{ix_model}"] = covariate_model.predict(df.loc[ix_tst, covariate_model.feature_names_])

100%|██████████| 10/10 [03:46<00:00, 22.69s/it]

CPU times: user 5min 27s, sys: 24.1 s, total: 5min 52s
Wall time: 3min 46s


In [11]:
%%time

# We set the maximum difference of two houses to be considered similar at $1k
max_dist = 1_000
pairs = pd.DataFrame(columns=pd.MultiIndex.from_tuples([("ModelId", "")]))
enum = 0


# Take 5,000 pairs
while enum < 5_000:

    # Pick model at random
    ix_model = np.random.choice(range(len(full_models)))
    covariate_model = covariate_models[ix_model]
    full_model = full_models[ix_model]

    df_tst = df.loc[df.loc[:, f"IsTest_{ix_model}"]==True, :].copy()

    # Pick one house at random from the test set
    first = np.random.choice(df_tst.index)

    # Find all the houses that are similar to the 1st house
    first_covariate_model_prediction = df_tst.loc[first, f"{covariate_model_prediction}_{ix_model}"]
    is_similar = (
        df_tst
        .drop(first)[f"{covariate_model_prediction}_{ix_model}"]
        .between(first_covariate_model_prediction-max_dist/2, first_covariate_model_prediction+max_dist/2)
    )

    # If there is not at least 1 similar house, skip
    if sum(is_similar) == 0:
        continue

    # Pick one house at random among the similar ones
    second = is_similar[is_similar].sample().index[0]

    # Save relevant information about the current pair
    pairs.loc[enum, ("ModelId", "")] = ix_model
    pairs.loc[enum, ("SameDecision", "")] = (df_tst.loc[first, decision_variables] == df_tst.loc[second, decision_variables]).all()

    for metric_name, metric in zip(
        ["HouseId", covariate_model_prediction, "Decision", full_model_prediction, "Actual"],
        ["HouseId", f"{covariate_model_prediction}_{ix_model}"] + decision_variables + [f"{full_model_prediction}_{ix_model}", target]
    ):
        for first_or_second_name, first_or_second in zip(["1st", "2nd"], [first, second]):
            pairs.loc[enum, (metric_name, first_or_second_name)] = df_tst.loc[first_or_second, :][metric]

    for prediction_name, prediction in zip(
        ["CovariateModel", "FullModel"],
        [f"{covariate_model_prediction}_{ix_model}", f"{full_model_prediction}_{ix_model}"]
    ):
        pairs.loc[enum, ("MeanAbsoluteError", prediction_name)] = mean_absolute_error(
            df_tst.loc[[first, second], :][target], df_tst.loc[[first, second], :][prediction]
        )

    enum += 1


# Beautify dataframe
pairs_display = pairs.map(beautify_int)

display(pairs_display)

CPU times: user 1min 23s, sys: 74.6 ms, total: 1min 23s
Wall time: 1min 27s


In [13]:
# For illustrative purpose, cherry-pick a single pair and pivot

eligible_pair = (
    (pairs.loc[:,("Actual","1st")] > 200_000)
    & (pairs.loc[:,("Decision","1st")] < pairs.loc[:,("Decision","2nd")])
    & (pairs.loc[:,("FullModel","1st")] < pairs.loc[:,("FullModel","2nd")])
    & (pairs.loc[:,("FullModel","1st")] < pairs.loc[:,("CovariateModel","1st")])
    & (pairs.loc[:,("FullModel","2nd")] > pairs.loc[:,("CovariateModel","2nd")])
    & (pairs.loc[:,("FullModel","1st")] > pairs.loc[:,("Actual","1st")])
    & (pairs.loc[:,("FullModel","2nd")] < pairs.loc[:,("Actual","2nd")])
    & (abs(pairs.loc[:,("FullModel","1st")] - pairs.loc[:,("Actual","1st")]) < (abs(pairs.loc[:,("CovariateModel","1st")] - pairs.loc[:,("Actual","1st")])))
    & (abs(pairs.loc[:,("FullModel","2nd")] - pairs.loc[:,("Actual","2nd")]) < (abs(pairs.loc[:,("CovariateModel","2nd")] - pairs.loc[:,("Actual","2nd")])))
    & (pairs.loc[:,("MeanAbsoluteError","FullModel")] <= pairs.loc[:,("MeanAbsoluteError","CovariateModel")] * .75)
)
pair_id = eligible_pair[eligible_pair].sample().index[0]
single_pair = pd.DataFrame(index=["1st", "2nd", "", "MeanAbsoluteError"])

for metric_name in ["HouseId", "CovariateModel", "Decision", "FullModel", "Actual"]:
    for first_or_second_name in ["1st", "2nd"]:
        single_pair.loc[first_or_second_name, metric_name] = pairs.loc[pair_id, (metric_name, first_or_second_name)]

    if metric_name in ["CovariateModel", "FullModel"]:
        single_pair.loc["MeanAbsoluteError", metric_name] = pairs.loc[pair_id, ("MeanAbsoluteError", metric_name)]

single_pair_display = single_pair.map(beautify_int)

display(single_pair_display)

In [15]:
print(f"Number of data points: {len(df):,.0f}")
print(f"N samples in training: {df[f'IsTest_{ix_model}'].isna().sum():,.0f}")
print(f"N samples in test: {df[f'IsTest_{ix_model}'].sum():,.0f}")

Number of data points: 1,460
N samples in training: 1,168
N samples in test: 292


In [16]:
no_skill_model_maes = []
covariate_model_maes = []
full_model_maes = []

for ix_model in range(len(full_models)):
    df_tst = df.loc[df.loc[:, f"IsTest_{ix_model}"]==True, :].copy()
    no_skill_model_maes.append(mean_absolute_error(df_tst[target], [df_tst[target].mean()] * len(df_tst)))
    covariate_model_maes.append(mean_absolute_error(df_tst[target], df_tst[f"{covariate_model_prediction}_{ix_model}"]))
    full_model_maes.append(mean_absolute_error(df_tst[target], df_tst[f"{full_model_prediction}_{ix_model}"]))

no_skill_model_mae = np.mean(no_skill_model_maes)
covariate_model_mae = np.mean(covariate_model_maes)
full_model_mae = np.mean(full_model_maes)

print(f"MAE No-Skill Model: {no_skill_model_mae:,.0f}")
print(f"MAE Covariate Model: {covariate_model_mae:,.0f}")
print(f"MAE Full Model: {full_model_mae:,.0f}")
print(f"Gain in MAE: {full_model_mae / covariate_model_mae - 1:.0%}")

MAE No-Skill Model: 57,641
MAE Covariate Model: 19,434
MAE Full Model: 17,753
Gain in MAE: -9%


In [17]:
feature_importances = sum([pd.Series(full_model.feature_importances_, index=full_model.feature_names_) for full_model in full_models]) / len(full_models)
feature_importances.sort_values(ascending=False).apply(lambda x: f"{x:.2f}%").rename("FeatureImportance")

display(feature_importances)

,0
LotArea,7.150819
MasVnrArea,3.068070
GrLivArea,21.882716
GarageArea,8.349466
PoolArea,0.126397
1stFlrSF,8.392583
2ndFlrSF,4.219289
TotalBsmtSF,13.206814
Foundation,4.092962
YearBuilt,15.404202


In [18]:
display(single_pair_display)

,HouseId,CovariateModel,Decision,FullModel,Actual
1st,806,"251,185",5,"240,444","227,680"
2nd,889,"251,588",6,"257,040","268,000"
,,,,,
MeanAbsoluteError,,"19,958",,"11,861",


In [19]:
pairs_display.head().drop([("SameDecision", ""), ("ModelId", "")], axis=1)

HouseId        CovariateModel          Decision     FullModel           \
      1st    2nd            1st      2nd      1st 2nd       1st      2nd   
0   1,369  1,298        143,050  143,115        5   5   141,787  142,967   
1   1,120  1,405        132,323  132,409        5   4   122,423  108,619   
2     834     34        152,442  152,164        6   5   163,381  147,648   
3     440    548        119,003  119,013        8   7   130,875  126,683   
4   1,408  1,099        112,097  111,906        5   6   103,713  114,172   

    Actual          MeanAbsoluteError            
       1st      2nd    CovariateModel FullModel  
0  144,000  140,000             2,032     2,589  
1  133,700  105,000            14,393     7,447  
2  167,000  165,500            13,946    10,734  
3  110,000  129,500             9,745    11,845  
4  112,000  128,000             8,095    11,056

In [20]:
is_duplicated_pair = pairs.loc[:, [("ModelId", ""), ("HouseId","1st"), ("HouseId","2nd")]].duplicated()
print(f"Number of pairs: {len(pairs):,.0f}")
print(f"Number of duplicated pairs: {sum(is_duplicated_pair):,.0f}")
print(f"Number of non-duplicated pairs: {len(pairs) - sum(is_duplicated_pair):,.0f}")

Number of pairs: 5,000
Number of duplicated pairs: 2,280
Number of non-duplicated pairs: 2,720


In [21]:
matching = pairs.groupby(("SameDecision", "")).apply(
    lambda d: pd.Series(
        {
            "Number of pairs": f'{len(d):,.0f}',
            "MAE Covariate Model": f'{d.loc[:, ("MeanAbsoluteError","CovariateModel")].mean():,.0f}',
            "MAE Full Model": f'{d.loc[:, ("MeanAbsoluteError","FullModel")].mean():,.0f}',
            "Gain in MAE": f'{
                d.loc[:, ("MeanAbsoluteError","FullModel")].mean() / d.loc[:, ("MeanAbsoluteError","CovariateModel")].mean() - 1:.0%}',
        }
    ),
    include_groups=False,
).rename({True: "Same decision", False: "Different decision"}).sort_index(ascending=False)
matching.index.name = None

display(matching)

,Number of pairs,MAE Covariate Model,MAE Full Model,Gain in MAE
Same decision,"1,908","15,304","14,942",-2%
Different decision,"3,092","17,759","15,423",-13%
